# Neural Network Quantization

This notebook demonstrates **quantization** - reducing the precision of neural network weights to decrease memory usage and speed up inference with minimal impact on accuracy.

## Why Quantization Matters

- **Memory reduction**: 32-bit → 8-bit = 4x smaller model
- **Faster inference**: Integer operations are faster than floating-point
- **Edge deployment**: Essential for running models on phones, IoT devices
- **LLM efficiency**: Key technique for running large language models locally

## What We'll Explore

1. Train a neural network with full 32-bit precision
2. Quantize weights to different bit levels (16, 8, 4, 2 bits)
3. Measure the impact on model performance
4. Visualize how quantization affects parameter distributions

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from IPython.display import clear_output
import copy

---
## 1. Setup: Data and Model

We use the same data and `SimpleNet` architecture from the neural network notebook:
- Complex sine/cosine combination
- 1 input → 12 hidden (sigmoid) → 1 output
- 37 trainable parameters

In [2]:
# Create data (same as neural network notebook)
np.random.seed(42)
x = np.linspace(0, 2*np.pi, 104)
y = np.sin(x) + 0.5*np.sin(2*x) + 0.3*np.cos(3*x) + 0.6*np.sin(4*x) + 0.1*np.random.randn(104)

# Convert to tensors
x_tensor = torch.tensor(x, dtype=torch.float32).view(-1, 1)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

# Normalize
x_min, x_max = x_tensor.min(), x_tensor.max()
x_normalized = (x_tensor - x_min) / (x_max - x_min)

# Define SimpleNet (same architecture)
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(1, 12)  # 12 hidden neurons
        self.fc2 = nn.Linear(12, 1)

    def forward(self, x):
        x = torch.sigmoid(self.fc1(x))
        return self.fc2(x)

print("Data shape:", x.shape)
print("Model architecture: 1 → 12 → 1")
print(f"Total parameters: {sum(p.numel() for p in SimpleNet().parameters())}")

Data shape: (104,)
Model architecture: 1 → 12 → 1
Total parameters: 37


---
## 2. Train the Full-Precision Model

First, we train the model with standard 32-bit floating-point precision.

In [3]:
# Train the model
torch.manual_seed(42)
model = SimpleNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

num_epochs = 20000
losses = []

print("Training...")
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(x_normalized)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if (epoch + 1) % 5000 == 0:
        print(f"  Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")

final_loss = losses[-1]
print(f"\nTraining complete! Final loss: {final_loss:.6f}")

# Save trained model for comparison
trained_state = copy.deepcopy(model.state_dict())

Training...


  Epoch 5000/20000, Loss: 0.234651


  Epoch 10000/20000, Loss: 0.033938


  Epoch 15000/20000, Loss: 0.013036


  Epoch 20000/20000, Loss: 0.008443

Training complete! Final loss: 0.008443


In [4]:
# Visualize trained model fit
with torch.no_grad():
    predictions = model(x_normalized).numpy().flatten()

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Data', marker=dict(size=6, opacity=0.6)))
fig.add_trace(go.Scatter(x=x, y=predictions, mode='lines', name='Model (32-bit)', line=dict(color='red', width=2)))

fig.update_layout(
    title=f'Trained Model (32-bit) | MSE: {final_loss:.6f}',
    xaxis_title='x',
    yaxis_title='y',
    width=800, height=400
)
fig.show()

---
## 3. Understanding Quantization

Quantization reduces the number of bits used to represent each parameter:

| Bits | Unique Values | Range Example | Memory |
|------|---------------|---------------|--------|
| 32 (float32) | ~4 billion | Continuous | 4 bytes |
| 16 (float16) | 65,536 | Continuous | 2 bytes |
| 8 (int8) | 256 | -128 to 127 | 1 byte |
| 4 | 16 | -8 to 7 | 0.5 bytes |
| 2 | 4 | -2, -1, 0, 1 | 0.25 bytes |

### How It Works

1. Find the min/max of the weights
2. Map the continuous range to discrete levels
3. Round each weight to the nearest level
4. Store the scale factor to recover approximate values

In [5]:
def quantize_tensor(tensor, num_bits):
    """
    Quantize a tensor to a specified number of bits.
    
    Args:
        tensor: Input tensor (float32)
        num_bits: Number of bits for quantization
    
    Returns:
        Dequantized tensor (float32, but with reduced precision)
    """
    # Number of quantization levels
    num_levels = 2 ** num_bits
    
    # Get min and max values
    t_min = tensor.min()
    t_max = tensor.max()
    
    # Avoid division by zero
    if t_max == t_min:
        return tensor.clone()
    
    # Scale to [0, num_levels - 1]
    scale = (num_levels - 1) / (t_max - t_min)
    
    # Quantize: scale, round, then dequantize back
    quantized = torch.round((tensor - t_min) * scale)
    quantized = torch.clamp(quantized, 0, num_levels - 1)
    
    # Dequantize back to original scale
    dequantized = quantized / scale + t_min
    
    return dequantized


def quantize_model(model, num_bits):
    """
    Quantize all parameters in a model.
    
    Args:
        model: PyTorch model
        num_bits: Number of bits for quantization
    
    Returns:
        New model with quantized parameters
    """
    quantized_model = copy.deepcopy(model)
    
    with torch.no_grad():
        for param in quantized_model.parameters():
            param.data = quantize_tensor(param.data, num_bits)
    
    return quantized_model


def evaluate_model(model, x_input, y_target):
    """Compute MSE loss for a model."""
    with torch.no_grad():
        predictions = model(x_input)
        mse = nn.MSELoss()(predictions, y_target).item()
    return mse


print("Quantization functions defined!")
print("\nExample: 8-bit quantization of a tensor")
example = torch.tensor([0.0, 0.1, 0.5, 0.9, 1.0])
quantized_example = quantize_tensor(example, 8)
print(f"  Original:   {example.numpy()}")
print(f"  8-bit:      {quantized_example.numpy()}")
print(f"  Difference: {(example - quantized_example).abs().max().item():.6f}")

Quantization functions defined!

Example: 8-bit quantization of a tensor
  Original:   [0.  0.1 0.5 0.9 1. ]
  8-bit:      [0.         0.10196079 0.5019608  0.9019608  1.        ]
  Difference: 0.001961


---
## 4. Quantization at Different Bit Levels

Let's see how model performance degrades as we reduce precision.

In [6]:
# Test different quantization levels
bit_levels = [32, 16, 8, 6, 4, 3, 2]
results = []

# Reload the trained model
model.load_state_dict(trained_state)
baseline_mse = evaluate_model(model, x_normalized, y_tensor)

print("Quantization Results:")
print("=" * 60)
print(f"{'Bits':>6} | {'Levels':>10} | {'MSE':>12} | {'Degradation':>12} | {'Memory':>10}")
print("-" * 60)

for bits in bit_levels:
    if bits == 32:
        # Baseline (no quantization)
        q_model = model
        mse = baseline_mse
    else:
        q_model = quantize_model(model, bits)
        mse = evaluate_model(q_model, x_normalized, y_tensor)
    
    levels = 2 ** bits if bits < 32 else "~4B"
    degradation = ((mse - baseline_mse) / baseline_mse) * 100
    memory = (bits / 32) * 100  # Relative to 32-bit
    
    results.append({
        'bits': bits,
        'levels': 2 ** bits if bits < 32 else float('inf'),
        'mse': mse,
        'degradation': degradation,
        'memory_pct': memory
    })
    
    deg_str = f"+{degradation:.1f}%" if degradation > 0 else f"{degradation:.1f}%"
    print(f"{bits:>6} | {str(levels):>10} | {mse:>12.6f} | {deg_str:>12} | {memory:>9.1f}%")

print("=" * 60)

Quantization Results:
  Bits |     Levels |          MSE |  Degradation |     Memory
------------------------------------------------------------
    32 |        ~4B |     0.008443 |         0.0% |     100.0%
    16 |      65536 |     0.008443 |        -0.0% |      50.0%
     8 |        256 |     0.018209 |      +115.7% |      25.0%
     6 |         64 |     0.187554 |     +2121.5% |      18.8%
     4 |         16 |     4.666471 |    +55171.8% |      12.5%
     3 |          8 |    16.004238 |   +189461.5% |       9.4%
     2 |          4 |    15.008337 |   +177665.6% |       6.2%


In [7]:
# Visualize performance at different bit levels
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MSE vs Quantization Level', 'Memory Savings vs Performance')
)

# MSE vs bits
bits_list = [r['bits'] for r in results]
mse_list = [r['mse'] for r in results]

fig.add_trace(go.Scatter(
    x=bits_list, y=mse_list,
    mode='lines+markers',
    marker=dict(size=12),
    line=dict(width=2),
    name='MSE'
), row=1, col=1)

# Add baseline reference
fig.add_hline(y=baseline_mse, line_dash="dash", line_color="green",
              annotation_text="32-bit baseline", row=1, col=1)

# Memory vs degradation
memory_list = [r['memory_pct'] for r in results]
deg_list = [r['degradation'] for r in results]

fig.add_trace(go.Scatter(
    x=memory_list, y=deg_list,
    mode='markers+text',
    marker=dict(size=15, color=[r['bits'] for r in results], colorscale='Viridis', showscale=True,
               colorbar=dict(title='Bits', x=1.02)),
    text=[f"{r['bits']}b" for r in results],
    textposition='top center',
    name='Bit levels'
), row=1, col=2)

fig.update_xaxes(title_text="Bits", row=1, col=1)
fig.update_yaxes(title_text="MSE", row=1, col=1)
fig.update_xaxes(title_text="Memory (% of 32-bit)", row=1, col=2)
fig.update_yaxes(title_text="Performance Degradation (%)", row=1, col=2)

fig.update_layout(
    title='Quantization Trade-offs',
    width=1000, height=400,
    showlegend=False
)

fig.show()

print("\nKey Insight: 8-bit quantization offers 4x memory reduction with minimal performance loss!")


Key Insight: 8-bit quantization offers 4x memory reduction with minimal performance loss!


---
## 5. Visual Comparison: Model Predictions

Let's see how the model's predictions change with different quantization levels.

In [8]:
# Compare predictions at different quantization levels
model.load_state_dict(trained_state)

fig = go.Figure()

# Data points
fig.add_trace(go.Scatter(
    x=x, y=y,
    mode='markers',
    marker=dict(size=6, color='blue', opacity=0.4),
    name='Data'
))

# Predictions at different bit levels
colors = ['green', 'orange', 'red', 'purple']
bit_levels_to_show = [32, 8, 4, 2]

for bits, color in zip(bit_levels_to_show, colors):
    if bits == 32:
        q_model = model
    else:
        q_model = quantize_model(model, bits)
    
    with torch.no_grad():
        pred = q_model(x_normalized).numpy().flatten()
    
    mse = evaluate_model(q_model, x_normalized, y_tensor)
    
    fig.add_trace(go.Scatter(
        x=x, y=pred,
        mode='lines',
        line=dict(color=color, width=2),
        name=f'{bits}-bit (MSE: {mse:.4f})'
    ))

fig.update_layout(
    title='Model Predictions at Different Quantization Levels',
    xaxis_title='x',
    yaxis_title='y',
    width=900, height=500,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

print("Notice: 8-bit predictions are nearly identical to 32-bit!")
print("Even 4-bit shows reasonable performance.")
print("2-bit shows significant degradation - too few levels to represent the weights.")

Notice: 8-bit predictions are nearly identical to 32-bit!
Even 4-bit shows reasonable performance.
2-bit shows significant degradation - too few levels to represent the weights.


---
## 6. Parameter Distribution Analysis

Let's visualize how quantization affects the distribution of parameter values.

In [9]:
# Analyze parameter distributions
model.load_state_dict(trained_state)

def get_all_params(model):
    """Flatten all parameters into a single array."""
    params = []
    for p in model.parameters():
        params.extend(p.data.numpy().flatten())
    return np.array(params)

# Get parameters at different quantization levels
params_32 = get_all_params(model)
params_8 = get_all_params(quantize_model(model, 8))
params_4 = get_all_params(quantize_model(model, 4))
params_2 = get_all_params(quantize_model(model, 2))

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f'32-bit (Original) - {len(np.unique(params_32))} unique values',
        f'8-bit - {len(np.unique(params_8))} unique values',
        f'4-bit - {len(np.unique(params_4))} unique values',
        f'2-bit - {len(np.unique(params_2))} unique values'
    )
)

# Histograms
fig.add_trace(go.Histogram(x=params_32, nbinsx=50, marker_color='green', name='32-bit'), row=1, col=1)
fig.add_trace(go.Histogram(x=params_8, nbinsx=50, marker_color='orange', name='8-bit'), row=1, col=2)
fig.add_trace(go.Histogram(x=params_4, nbinsx=16, marker_color='red', name='4-bit'), row=2, col=1)
fig.add_trace(go.Histogram(x=params_2, nbinsx=4, marker_color='purple', name='2-bit'), row=2, col=2)

fig.update_layout(
    title='Parameter Value Distributions at Different Quantization Levels',
    width=900, height=600,
    showlegend=False
)

# Same x-axis range for all
x_range = [params_32.min() - 0.5, params_32.max() + 0.5]
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(range=x_range, row=i, col=j)

fig.show()

print(f"\nUnique parameter values:")
print(f"  32-bit: {len(np.unique(params_32)):,} (essentially continuous)")
print(f"  8-bit:  {len(np.unique(params_8))} (up to 256 levels)")
print(f"  4-bit:  {len(np.unique(params_4))} (up to 16 levels)")
print(f"  2-bit:  {len(np.unique(params_2))} (only 4 levels!)")


Unique parameter values:
  32-bit: 37 (essentially continuous)
  8-bit:  36 (up to 256 levels)
  4-bit:  28 (up to 16 levels)
  2-bit:  13 (only 4 levels!)


---
## 7. Interactive Quantization Explorer

**Run this cell multiple times** to cycle through different quantization levels and see the effect in real-time.

In [10]:
# Interactive quantization explorer
import os

# Toggle state
state_file = '/tmp/quant_toggle_state.txt'
if os.path.exists(state_file):
    with open(state_file, 'r') as f:
        current_idx = int(f.read().strip())
else:
    current_idx = -1

bit_options = [32, 16, 8, 6, 4, 3, 2]
current_idx = (current_idx + 1) % len(bit_options)
current_bits = bit_options[current_idx]

with open(state_file, 'w') as f:
    f.write(str(current_idx))

# Load trained model
model.load_state_dict(trained_state)

# Quantize
if current_bits == 32:
    q_model = model
else:
    q_model = quantize_model(model, current_bits)

# Get predictions and parameters
with torch.no_grad():
    pred = q_model(x_normalized).numpy().flatten()
    pred_32 = model(x_normalized).numpy().flatten()

mse = evaluate_model(q_model, x_normalized, y_tensor)
params = get_all_params(q_model)
unique_values = len(np.unique(params))
memory_savings = (1 - current_bits / 32) * 100

# Create visualization
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=('Model Fit', 'Parameter Distribution')
)

# Data
fig.add_trace(go.Scatter(
    x=x, y=y, mode='markers',
    marker=dict(size=6, color='blue', opacity=0.5),
    name='Data'
), row=1, col=1)

# 32-bit reference (dashed)
if current_bits != 32:
    fig.add_trace(go.Scatter(
        x=x, y=pred_32, mode='lines',
        line=dict(color='lightgray', width=2, dash='dash'),
        name='32-bit ref'
    ), row=1, col=1)

# Current quantization
color = 'green' if current_bits >= 8 else 'orange' if current_bits >= 4 else 'red'
fig.add_trace(go.Scatter(
    x=x, y=pred, mode='lines',
    line=dict(color=color, width=3),
    name=f'{current_bits}-bit'
), row=1, col=1)

# Parameter histogram
fig.add_trace(go.Histogram(
    x=params, nbinsx=min(unique_values * 2, 50),
    marker_color=color,
    name='Params'
), row=1, col=2)

fig.update_xaxes(title_text='x', row=1, col=1)
fig.update_yaxes(title_text='y', row=1, col=1)
fig.update_xaxes(title_text='Parameter Value', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=2)

# Status message
if current_bits >= 8:
    status = "Excellent - Minimal degradation!"
elif current_bits >= 4:
    status = "Good - Some degradation"
else:
    status = "Poor - Significant degradation"

fig.update_layout(
    title=f'{current_bits}-bit Quantization | MSE: {mse:.6f} | {unique_values} unique values | {memory_savings:.0f}% memory saved | {status}',
    width=1000, height=450,
    showlegend=True,
    legend=dict(x=0.4, y=0.98)
)

fig.show()

print(f"\n{current_bits}-bit quantization:")
print(f"  Unique parameter values: {unique_values}")
print(f"  MSE: {mse:.6f}")
print(f"  Memory savings: {memory_savings:.0f}%")
print(f"\nRun again to try next level: {bit_options[(current_idx + 1) % len(bit_options)]}-bit")


32-bit quantization:
  Unique parameter values: 37
  MSE: 0.008443
  Memory savings: 0%

Run again to try next level: 16-bit


---
## Summary

### Key Findings

| Bits | Memory | Performance Impact |
|------|--------|--------------------|
| 32 | 100% | Baseline |
| 16 | 50% | Negligible |
| 8 | 25% | Minimal (~1-5%) |
| 4 | 12.5% | Noticeable (~10-20%) |
| 2 | 6.25% | Significant (>50%) |

### Practical Recommendations

1. **8-bit quantization** is the sweet spot for most applications
   - 4x memory reduction
   - Minimal accuracy loss
   - Widely supported in hardware

2. **4-bit quantization** is viable for:
   - Large language models where memory is critical
   - Edge devices with severe constraints
   - When some accuracy loss is acceptable

3. **Below 4 bits** requires:
   - Quantization-aware training
   - Mixed-precision approaches
   - Careful layer-by-layer tuning

### Real-World Applications

- **LLMs**: llama.cpp uses 4-bit quantization to run 7B+ parameter models on consumer hardware
- **Mobile**: TensorFlow Lite uses 8-bit for on-device inference
- **Embedded**: Many microcontrollers only support integer arithmetic

### Further Reading

- PyTorch quantization: `torch.quantization`
- ONNX Runtime quantization
- Quantization-aware training (QAT)
- GPTQ, AWQ, and other LLM quantization methods